# 缓存

缓存保存数据和中间计算结果，减少重复读取与计算。本篇从读写数据、接入因子任务和恢复上次缓存三个场景介绍用法；类继承关系、文件组织和同步细节见[原理和细节](原理和细节.ipynb)。

## 选择缓存与配置启动方式

| 使用场景 | 推荐实现 | 导入位置 |
|---|---|---|
| 自定义 DataFrame 或时序数据读写 | `FeatherDTCache` | `QuantStudio.Core.FileCache` |
| 因子数据及衍生计算 | `FeatherFactorCache` | `QuantStudio.Factor.FactorCache` |

`Cache`、`DTCache`、`FactorCache` 定义接口；`FileDTCache`、`FileFactorCache` 提供文件组织层，仍需子类实现具体数据格式。日常使用优先选择上表的 Feather 实现。

| 参数 | 用法 |
|---|---|
| `CacheDir` | 已存在的缓存目录；要跨会话保留数据，应明确指定并提前创建 |
| `DTRuler` | 时点标尺；因子任务中应与上下文一致 |
| `StartMode="new"` | 默认值，启动时清空已有缓存 |
| `StartMode="continue"` | 恢复原缓存状态；缺少状态文件时会退回新建模式 |
| `PIDs` | 因子缓存的进程 ID 列表，应与并行上下文的 `PIDList` 配套 |

建议用 `with cache:` 管理生命周期：进入时启动，退出时保存状态。`cache.end(clear=True)` 用于清空并结束。未指定目录时使用临时目录，不适合作为跨会话缓存。

本篇用临时目录隔离演示，退出最外层 `TemporaryDirectory()` 后会删除示例数据。实际复用时应改用固定目录，例如先用 `Path("缓存目录").mkdir(parents=True, exist_ok=True)` 创建。

## 读写 DataFrame

普通数据使用 `writeData()` / `readData()`；时序数据使用 `writeDTData()` / `readDTData()`。两类方法位于不同的数据空间，同名键不会相互替代。

In [1]:
import logging
from QuantStudio.Core import setDefaultLogLevel

setDefaultLogLevel(logging.WARNING)

In [2]:
import datetime as dt
import tempfile
import pandas as pd
from QuantStudio.Core.FileCache import FeatherDTCache

dates = [dt.datetime(2025, 1, day) for day in range(1, 6)]
data = pd.DataFrame({"A": [1., 2., 3.], "B": [4., 5., 6.]}, index=dates[:3])

with tempfile.TemporaryDirectory() as cache_dir:
    with FeatherDTCache(args={"CacheDir": cache_dir, "DTRuler": dates}) as cache:
        cache.writeData("参数表", pd.DataFrame({"倍数": [2.]}), if_exists="replace")
        cache.writeDTData("示例行情", data, if_exists="replace")
        restored = cache.readDTData("示例行情")
        pd.testing.assert_frame_equal(restored, data)
        print(restored)

              A    B
2025-01-01  1.0  4.0
2025-01-02  2.0  5.0
2025-01-03  3.0  6.0


| 操作 | 说明 |
|---|---|
| `writeData(key, data, if_exists="replace")` | 替换一个普通数据项；`writeDTData` 对应时序空间 |
| `if_exists="append"`（默认） | 合并已有数据，重复索引保留旧记录；修正原值时使用 `"replace"` |
| `readData(key)` / `readDTData(key)` | 不存在时返回 `None` |
| `checkDataExistence(key)` / `checkDTDataExistence(key)` | 检查数据项是否存在 |
| `clearData(key)` / `clearDTData(key)` | 清理指定数据项；省略键会清空对应空间 |

对于需要按 Python 对象保存的表格，读写两端均指定 `data_type="object"`。普通数值表格通常不必指定。

## 查询已计算的时间范围

`getDTRange(key, dt_range)` 返回需要补充的范围，`None` 表示已覆盖。手动写入数据后，需要在确认写入成功时调用 `updateDTRange()` 登记范围；底层 `writeDTData()` **不会自动登记计算覆盖范围**。业务因子会在自己的计算流程中维护这些记录。

In [3]:
with tempfile.TemporaryDirectory() as cache_dir:
    with FeatherDTCache(args={"CacheDir": cache_dir, "DTRuler": dates}) as cache:
        requested_range = (dates[0], dates[2])
        assert cache.getDTRange("示例行情", requested_range) == requested_range
        cache.writeDTData("示例行情", data, if_exists="replace")
        cache.updateDTRange("示例行情", requested_range)
        assert cache.getDTRange("示例行情", requested_range) is None
        print("请求范围已被缓存覆盖")

请求范围已被缓存覆盖


## 恢复上次手动写入的数据

同一个目录以 `"new"` 写入并正常结束后，可用新的缓存对象以 `"continue"` 恢复。这里手动指定的键始终是 `"示例行情"`，不涉及 `DataVersion`；只有节点自己构造的结果缓存键才可能包含版本。

In [4]:
with tempfile.TemporaryDirectory() as cache_dir:
    common_args = {"CacheDir": cache_dir, "DTRuler": dates}
    with FeatherDTCache(args=common_args | {"StartMode": "new"}) as cache:
        cache.writeDTData("示例行情", data, if_exists="replace")
        cache.updateDTRange("示例行情", (dates[0], dates[2]))

    with FeatherDTCache(args=common_args | {"StartMode": "continue"}) as cache:
        restored = cache.readDTData("示例行情")
        pd.testing.assert_frame_equal(restored, data)
        assert cache.getDTRange("示例行情", (dates[0], dates[2])) is None
        print("数据与覆盖范围均已恢复")

数据与覆盖范围均已恢复


## 接入因子计算并复用上次结果

因子任务使用 `FeatherFactorCache`，通过构造 `FactorContext(DataCache=cache, ...)` 接入。缓存需要在 `engine.run()` 前启动，因子自身也需要启用缓存；仅为基础 `Node` 提供 `DataCache` 不会自动缓存返回值。

普通因子结果缓存使用计算定义、有效截面、完整时点标尺以及数据版本构造键。跨运行复用需要同时满足：

1. 缓存保留；重新创建缓存对象时使用原目录和 `StartMode="continue"`。
2. 显式设置相同且非空的 `DataVersion`，表示使用同一份数据快照。
3. 因子定义、有效截面、完整时点标尺及相关分片配置保持兼容。
4. 缓存覆盖所需日期；缺失部分仍需计算。

`DataVersion` 是调用方声明的标签，不会自动创建数据快照、切换数据库版本或检测数据更新。数据修正后应更换标签。未设置时使用本次 `RunID`，普通因子结果不能跨运行命中；完整截面节点的共享结果缓存始终按本次运行隔离，详见[原理和细节](原理和细节.ipynb)。

下面使用本地数据创建一个乘二因子：第一轮写入，第二轮创建新的缓存和上下文恢复结果。运算函数会记录真正执行的次数，用于确认第二轮没有重复计算。

In [5]:
from QuantStudio.Core.api import Engine
from QuantStudio.Factor.Factor import DataFactor, FactorContext, FactorLocalContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.Factor.FactorOperation import makeFactorOperator

ids = list(data.columns)
source = DataFactor(data, args={"Name": "示例数据"})
calculation_calls = []


def double_values(factor, idt, iid, inputs, args):
    """将输入数值乘以二。

    Args:
        factor: 当前因子。
        idt: 当前时点或时点序列。
        iid: 当前资产或资产序列。
        inputs: 输入数据数组列表。
        args: 算子参数。

    Returns:
        乘以二后的数组。
    """
    calculation_calls.append(1)
    return inputs[0] * 2


operator = makeFactorOperator(double_values, "Point", args={"DTMode": "多时点", "IDMode": "多ID"})
factor = operator(source)
engine = Engine()

with tempfile.TemporaryDirectory() as cache_dir:
    common_args = {"CacheDir": cache_dir, "DTRuler": dates, "PIDs": ["0"]}
    for mode in ("new", "continue"):
        with FeatherFactorCache(args=common_args | {"StartMode": mode}) as cache:
            context = FactorContext(
                DTRuler=dates, SectionIDs=ids, DataCache=cache,
                DataVersion="示例行情-v1",
            )
            request = FactorLocalContext(DTs=dates[:3], IDs=ids)
            result = engine.run([factor], context, fwd_data_list=[request])[0]
            pd.testing.assert_frame_equal(result, data * 2, check_freq=False)
            if mode == "new":
                first_call_count = len(calculation_calls)
                assert first_call_count > 0
            else:
                assert len(calculation_calls) == first_call_count
            print(mode, "累计实际计算次数:", len(calculation_calls))

new 累计实际计算次数: 1
continue 累计实际计算次数: 1


如果只在同一会话内连续执行，可以保持同一个缓存对象处于启动状态，在 `with cache:` 内多次调用 `run()`；要让普通因子跨轮命中，仍需相同的非空 `DataVersion`。

## 缓存未命中时检查什么

| 现象 | 优先检查 |
|---|---|
| 每轮都重新计算 | 是否省略了 `DataVersion`；节点是否支持并启用缓存 |
| 重启后缓存消失 | 是否使用临时目录；是否以默认 `"new"` 重新启动 |
| 使用 `"continue"` 仍重新开始 | 状态文件是否保留，目录和状态文件设置是否一致 |
| 修改请求后重新计算 | 定义、截面或完整时点标尺改变会改变结果键；仅请求日期范围改变时还要看覆盖范围 |
| 更新数据后读到旧值 | 是否沿用了旧 `DataVersion`；手动写入时是否使用了保留旧索引的 `"append"` |

底层文件布局、原始数据与结果缓存的区别、覆盖区间记录和多进程同步见[原理和细节](原理和细节.ipynb)。